# SQLite Database Setup

These scripts are intended to be run initially to ensure the structure of the Database is created correctly.

This means:

- Creating the Database itself
- Creating all Tables on the Database
    - Ensuring each table has Foreign Keys created
    - Ensuring all tables have necessary indexes created

## Set Local Variables

In [2]:
import json
import pandas as pd

db_name = 'DiscordChatMusic.db'
local_table_definitions = 'sqlite_resources/DDL/TABLE_CREATION/'

# Order of table creation is manually designed. This is intentional as this is a Star Schema setup. So Dimensions must be created before Facts
local_tables = [
    'discord_logging.sql',          # Just for logging
    'discord_guilds.sql',
    'discord_channels.sql',         # References Guilds
    'discord_authors.sql',
    'discord_author_roles.sql',     # References Authors
    'discord_author_nicknames.sql', # References Authors
    'discord_messages.sql',         # References Authors / Channels / Guilds
    'discord_mentions.sql',         # References Authors / Messages
    'discord_reaction_authors.sql', # References Authors / Messages
    'discord_attachments.sql',      # References Messages
    'discord_reactions.sql',        # References Messages
    'discord_stickers.sql',         # References Messages
    'discord_urls.sql',             # References Messages
    'discord_embeds.sql'            # References Messages
]

json_file = "tunes.json"

with open(json_file, 'r') as j:
    contents = json.loads(j.read())

In [3]:
import sqlite3

# Creates a new database if doesn't exist
conn = sqlite3.connect(db_name)

for local_table in local_tables:
    file_url = local_table_definitions + local_table
    with open(file_url, 'r') as sql_file:
        try:
            conn.executescript(sql_file.read())
        except sqlite3.OperationalError:
            print(f"OperationalError: On {local_table} at location {file_url}")

#conn.close()

# Filling Tables

In [4]:
from typing import List

def comma_list_surrounded_by_character(l: List[str], c: str):

    return c + f"{c},{c}".join(l) + c

def values_in_table(table: str, filter_col: str, filter_condition: List[str], conn: sqlite3.Connection, not_flag=False) -> bool:

    cur = conn.cursor()

    if not_flag:
        not_string = 'NOT'
    else:
        not_string = ''

    select_sql = f"""
        SELECT 
            {filter_col}
        FROM {table}
        WHERE {filter_col} 
            -- Take a Python list and turn it into a list of SQL strings
            {not_string} IN ({comma_list_surrounded_by_character(l=filter_condition,c="'")})
    """

    cur.execute(select_sql)

    result = cur.fetchall()

    #print(result)

    cur.close()

    # No matches? Return -1
    if len(result) == 0:
        return -1

    # Convert tuple -> List
    return [i[0] for i in result]

In [5]:
from datetime import datetime

def todays_date():

    return datetime.today().strftime('%Y-%m-%d')

def logging_process():

    return "sqlite_database_setup.ipynb"

def sqlite_insert_log_single_success(conn: sqlite3.Connection, table_name: str, logging_table: str, logging_table_id: str, file_name: str):

    cur = conn.cursor()

    insert_sql = f"""
    INSERT INTO {table_name}(logging_date, logging_table, logging_table_id, logging_file_name, logging_process)
    VALUES('{todays_date()}', '{logging_table}','{logging_table_id}','{file_name}','{logging_process()}');
    """

    cur.execute(insert_sql)

    conn.commit()
    cur.close()

    return

def sqlite_insert_single_row_df(json_blob: str, inner_key: str, mapping: dict, table_name: str, table_primary_key: str, conn: sqlite3.Connection, file_name: str) -> None:

    df = pd.DataFrame([json_blob[inner_key]])

    assert len(df) == 1, f"sqlite_insert_single_row_df is designed for single items. This instead has {len(df)} items."

    # Filter DataFrame to only be mapped columns, if the col doesn't exist in the Database we are not interested.
    df = df[mapping.keys()]

    df.rename(columns=mapping, inplace=True)

    ids = df[table_primary_key].to_list()

    table_values = values_in_table(
        table=table_name,
        filter_col=table_primary_key,
        filter_condition=ids,
        conn=conn, 
        not_flag=False
    )

    id = ids[0]

    # Single value not inserted into table on DB
    if table_values == -1:
        df.to_sql(table_name,conn,if_exists='append',index=False)
        sqlite_insert_log_single_success(
            conn=conn,
            table_name='discord_logging',
            logging_table=table_name,
            file_name=file_name,
            logging_table_id=id
        )
        print(f"{df.head(1)} inserted into {table_name} successfully.")
    else:
        print(f"{table_values} already exists within {table_name}, nothing inserted.")

    return id

## discord_guilds

In [6]:
guild_id = sqlite_insert_single_row_df(
    json_blob=contents,
    inner_key="guild",
    mapping={
        "id": "guild_id",
        "name": "guild_name",
        "iconUrl": "guild_icon_URL"
    },
    table_name='discord_guilds',
    table_primary_key='guild_id',
    conn=conn,
    file_name=json_file
)

['1185399888880619550'] already exists within discord_guilds, nothing inserted.


## discord_channels

In [7]:
channel_id = sqlite_insert_single_row_df(
    json_blob=contents,
    inner_key="channel",
    table_name='discord_channels',
    table_primary_key='channel_id',
    mapping={
        "id": "channel_id",
        "type": "channel_type",
        "categoryId": "guild_id",
        "category": "channel_category",
        "name": "channel_name"
    },
    conn=conn,
    file_name=json_file
)

['1203039998740271166'] already exists within discord_channels, nothing inserted.


## discord_messages

In [ ]:
def sqlite_insert_log_multiple_success(conn: sqlite3.Connection, table_name: str, logging_table: str, logging_table_ids: List[str], file_name: str):

    num_ids = len(logging_table_ids)

    df_dict = {
        "logging_date": [todays_date()] * num_ids,
        "logging_table": [logging_table] * num_ids,
        "logging_table_id": logging_table_ids,
        "logging_file_name": [file_name] * num_ids,
        "logging_process": [logging_process()] * num_ids
    }

    insert_df = pd.DataFrame(df_dict)

    insert_df.to_sql(table_name,conn,if_exists='append',index=False)

    return

def insert_df_to_db(df: pd.DataFrame, table_name: str, logging_table: str, conn: sqlite3.Connection, file_name: str, ids: List[str]) -> None:
    df.to_sql(table_name,conn,if_exists='append',index=False)
    sqlite_insert_log_multiple_success(
        conn=conn,
        table_name=table_name,
        logging_table=logging_table,
        file_name=file_name,
        logging_table_ids=ids
    )
    return

def sqlite_insert_multiple_row_df(json_blob: str, inner_key: str, mapping: dict, table_name: str, table_primary_key: str, conn: sqlite3.Connection, file_name: str,sep="_",constant_cols={}) -> None:

    df = pd.json_normalize(
        data=json_blob[inner_key],
        sep=sep
    )

    for key, value in constant_cols.items():
        df[key] = value

    # Filter DataFrame to only be mapped columns, if the col doesn't exist in the Database we are not interested.
    df = df[mapping.keys()]

    df.rename(columns=mapping, inplace=True)

    ids = df[table_primary_key].to_list()

    table_values = values_in_table(
        table=table_name,
        filter_col=table_primary_key,
        filter_condition=ids,
        conn=conn, 
        not_flag=False
    )

    # ALL values are not present on DataBase, we are good to insert everything
    # Unecessary to check because we are attempting to insert as a default, exiting early only if there is nothing to insert
    #if table_values == -1:

    # At-least one record from the DataFrame exists within the DataBase
    if table_values != -1:
        # Filter to records that are not present in the DataBase
        df = df[~df[table_primary_key].isin(table_values)]
    # Nothing to insert, leave
    if df.empty:
        return ids
    
    insert_df_to_db(
        df=df,
        conn=conn,
        table_name=table_name,
        logging_table='discord_logging',
        file_name=file_name,
        ids=ids
    )

    return ids

sqlite_insert_multiple_row_df(
    json_blob=contents,
    inner_key="messages",
    mapping={
        "id": "message_id",
        "type": "message_type",
        "timestamp": "message_timestamp",
        "content": "message_content",
        "author_id": "author_id",
        "guild_id": "guild_id",
        "channel_id": "channel_id"
    },
    constant_cols={
        "guild_id": guild_id,
        "channel_id": channel_id
    },
    table_primary_key="message_id",
    table_name='discord_messages',
    conn=conn,
    file_name=json_file
)


In [10]:
sqlite_insert_multiple_row_df(
    json_blob=contents,
    # Because of json_normalize, authors are at the messages level
    inner_key="messages",
    mapping={
        "author_id": "author_id",
        "author_name": "author_name",
        "author_isBot": "author_is_bot",
        "author_avatarUrl": "author_avatar_URL",
    },
    table_primary_key="author_id",
    table_name='discord_authors',
    conn=conn,
    file_name=json_file
)

IntegrityError: UNIQUE constraint failed: discord_authors.author_id